# GK-SMOTE Demo

This notebook demonstrates how to use **GK-SMOTE** for imbalanced binary classification.

We will:

1. generate a synthetic imbalanced dataset,
2. visualize the class imbalance,
3. apply **GK-SMOTE**,
4. compare performance against:
   - no resampling,
   - standard SMOTE,
   - GK-SMOTE,
5. evaluate using imbalance-aware metrics.

---

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from gksmote import GKSMOTE
from gksmote.metrics import evaluate_binary_classification

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.25
K_NEIGHBORS = 5

## 1. Generate an imbalanced dataset

In [ ]:
X, y = make_classification(
    n_samples=1500,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    n_repeated=0,
    n_clusters_per_class=2,
    weights=[0.92, 0.08],
    flip_y=0.08,
    class_sep=1.0,
    random_state=RANDOM_STATE,
)

print("Original class distribution:")
print(Counter(y))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Training distribution:", Counter(y_train))
print("Test distribution:", Counter(y_test))

## 2. Visualize the original training data

To make visualization possible, we project the data into 2D using PCA.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_2d = pca.fit_transform(X_train)

plt.figure(figsize=(8, 6))
plt.scatter(
    X_train_2d[y_train == 0, 0],
    X_train_2d[y_train == 0, 1],
    alpha=0.6,
    label="Majority Class",
)
plt.scatter(
    X_train_2d[y_train == 1, 0],
    X_train_2d[y_train == 1, 1],
    alpha=0.8,
    label="Minority Class",
)
plt.title("Original Training Data (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()

## 3. Baseline model without resampling

In [ ]:
baseline_clf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
baseline_clf.fit(X_train, y_train)

y_pred_base = baseline_clf.predict(X_test)
y_score_base = baseline_clf.predict_proba(X_test)[:, 1]

baseline_results = evaluate_binary_classification(y_test, y_pred_base, y_score_base)
baseline_results

## 4. Standard SMOTE

In [ ]:
smote = SMOTE(k_neighbors=K_NEIGHBORS, random_state=RANDOM_STATE)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print(Counter(y_smote))

In [ ]:
smote_clf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
smote_clf.fit(X_smote, y_smote)

y_pred_smote = smote_clf.predict(X_test)
y_score_smote = smote_clf.predict_proba(X_test)[:, 1]

smote_results = evaluate_binary_classification(y_test, y_pred_smote, y_score_smote)
smote_results

## 5. GK-SMOTE

In [ ]:
gksmote = GKSMOTE(k_neighbors=K_NEIGHBORS, random_state=RANDOM_STATE)
X_gk, y_gk = gksmote.fit_resample(X_train, y_train)

print("After GK-SMOTE:")
print(Counter(y_gk))

print("\nGK-SMOTE diagnostics:")
print("Minority class:", gksmote.minority_class_)
print("Majority class:", gksmote.majority_class_)
print("Removed noisy minority samples:", gksmote.n_noise_removed_)
print("Retained minority samples:", gksmote.n_retained_minority_)
print("Generated synthetic samples:", gksmote.n_generated_)

In [ ]:
gk_clf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
gk_clf.fit(X_gk, y_gk)

y_pred_gk = gk_clf.predict(X_test)
y_score_gk = gk_clf.predict_proba(X_test)[:, 1]

gk_results = evaluate_binary_classification(y_test, y_pred_gk, y_score_gk)
gk_results

## 6. Visualize resampled data

Again, we use PCA for 2D projection.

In [ ]:
X_smote_2d = pca.transform(X_smote)
X_gk_2d = pca.transform(X_gk)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(
    X_smote_2d[y_smote == 0, 0],
    X_smote_2d[y_smote == 0, 1],
    alpha=0.5,
    label="Majority",
)
axes[0].scatter(
    X_smote_2d[y_smote == 1, 0],
    X_smote_2d[y_smote == 1, 1],
    alpha=0.7,
    label="Minority + Synthetic",
)
axes[0].set_title("SMOTE Resampled Data")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].legend()

axes[1].scatter(
    X_gk_2d[y_gk == 0, 0],
    X_gk_2d[y_gk == 0, 1],
    alpha=0.5,
    label="Majority",
)
axes[1].scatter(
    X_gk_2d[y_gk == 1, 0],
    X_gk_2d[y_gk == 1, 1],
    alpha=0.7,
    label="Minority + Synthetic",
)
axes[1].set_title("GK-SMOTE Resampled Data")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Compare results

In [ ]:
results_df = pd.DataFrame(
    [
        {"Method": "Baseline", **baseline_results},
        {"Method": "SMOTE", **smote_results},
        {"Method": "GK-SMOTE", **gk_results},
    ]
)

results_df

In [ ]:
results_df.set_index("Method").plot(kind="bar", figsize=(10, 6))
plt.title("Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="best")
plt.show()

## 8. Conclusion

This notebook shows the full workflow for using **GK-SMOTE**:

- generate imbalanced data,
- resample with GK-SMOTE,
- train a classifier,
- evaluate using imbalance-aware metrics,
- compare against baseline and SMOTE.

You can now replace the synthetic dataset with your own real dataset for further experiments.